# Final Data Science Capstone Project
## Predicting Air Quality Index (AQI) from Pollutant Concentrations Across Indian Monitoring Stations

**Dataset:** `station_day_clean.csv` (cleaned station-level daily air-quality data, 2015-2020)

**Notebook flow:** 1) Problem definition -> 2) Data loading & checks -> 3) Feature engineering -> 4) EDA -> 5) Modelling (regression) -> 6) Evaluation -> 7) Error analysis & robustness -> 8) Classification of AQI category -> 9) Insights & export for dashboard

## 1. Problem Definition

**Problem statement:** Air pollution is a major public-health problem in India. The Air Quality Index (AQI) summarises pollution into one number, but it is only available after all pollutant readings are processed. Can we *predict* the AQI of a station-day accurately from pollutant readings and calendar/location information, and which factors drive it most?

**Objectives**
- Predict daily AQI (regression) and AQI category (classification).
- Identify the pollutants, seasons and states that influence AQI the most.
- Compare several models using R2, MAE and RMSE and analyse where they fail.

**Scope:** 109 stations, 21 states, Jan 2015 - Jul 2020, 11 pollutants. Not in scope: weather variables, real-time forecasting.

In [ ]:
import warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             accuracy_score, f1_score, confusion_matrix, classification_report)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RS = 42                      # random seed for reproducibility
R = {}                       # dictionary to collect results
import os; os.makedirs("figs", exist_ok=True)

## 2. Data Loading & Quality Check

In [ ]:
df = pd.read_csv("station_day_clean.csv")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
print("Shape:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Stations:", df["StationId"].nunique(), "| States:", df["State"].nunique())
print("Missing values:", int(df.isnull().sum().sum()), "| Duplicate rows:", int(df.duplicated().sum()))
R["rows"], R["cols"] = int(df.shape[0]), int(df.shape[1])
R["stations"], R["states"] = int(df["StationId"].nunique()), int(df["State"].nunique())
R["date_min"], R["date_max"] = str(df["Date"].min().date()), str(df["Date"].max().date())
R["missing"], R["dups"] = int(df.isnull().sum().sum()), int(df.duplicated().sum())
df.head()

In [ ]:
df.describe().T.round(2)

## 3. Feature Engineering

In [ ]:
data = df.copy()
data["DayOfWeek"]  = data["Date"].dt.dayofweek
data["IsWeekend"]  = (data["DayOfWeek"] >= 5).astype(int)
data["DayOfYear"]  = data["Date"].dt.dayofyear
data["Month_sin"]  = np.sin(2 * np.pi * data["Month"] / 12)      # cyclic month encoding
data["Month_cos"]  = np.cos(2 * np.pi * data["Month"] / 12)
data["PM_ratio"]   = data["PM2.5"] / data["PM10"].replace(0, np.nan)   # fine vs coarse particles
data["NOx_NO2_gap"] = data["NOx"] - data["NO2"]
data["PM_ratio"] = data["PM_ratio"].fillna(data["PM_ratio"].median())

pollutants = ["PM2.5","PM10","NO","NO2","NOx","NH3","CO","SO2","O3","Benzene","Toluene"]
num_feats  = pollutants + ["PM_ratio","NOx_NO2_gap","Year","Month_sin","Month_cos","DayOfWeek","IsWeekend","DayOfYear"]
X = pd.concat([data[num_feats],
               pd.get_dummies(data["Season"], prefix="Season", dtype=int),
               pd.get_dummies(data["State"],  prefix="State",  dtype=int)], axis=1)
y = data["AQI"]
print("Feature matrix:", X.shape)
R["n_features"] = int(X.shape[1])

## 4. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["AQI"], bins=40, kde=True, color="#2a6f97", ax=ax[0])
ax[0].set_title("Distribution of AQI"); ax[0].set_xlabel("AQI")
order = ["Good","Satisfactory","Moderate","Poor","Very Poor","Severe"]
pal = ["#2e9e4f","#8cc152","#f2c94c","#f2994a","#eb5757","#8b1c1c"]
cnt = df["AQI_Bucket"].value_counts().reindex(order)
ax[1].bar(order, cnt.values, color=pal); ax[1].set_title("Records per AQI category")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("figs/01_aqi_distribution.png", bbox_inches="tight"); plt.show()
R["aqi_mean"], R["aqi_median"] = float(df.AQI.mean()), float(df.AQI.median())
R["aqi_std"], R["aqi_max"] = float(df.AQI.std()), float(df.AQI.max())
R["bucket_counts"] = cnt.to_dict()
R["bucket_pct"] = (cnt / cnt.sum() * 100).round(1).to_dict()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
yr = df.groupby("Year")["AQI"].mean()
ax[0].plot(yr.index, yr.values, marker="o", color="#2a6f97"); ax[0].set_title("Average AQI by Year")
ax[0].set_xlabel("Year"); ax[0].set_ylabel("Mean AQI")
mo = df.groupby("Month")["AQI"].mean()
ax[1].bar(mo.index, mo.values, color="#2a6f97"); ax[1].set_title("Average AQI by Month")
ax[1].set_xlabel("Month"); ax[1].set_xticks(range(1, 13))
plt.tight_layout(); plt.savefig("figs/02_year_month_trend.png", bbox_inches="tight"); plt.show()
R["aqi_by_year"] = yr.round(1).to_dict()
R["aqi_by_month"] = mo.round(1).to_dict()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
se = df.groupby("Season")["AQI"].mean().sort_values()
ax[0].barh(se.index, se.values, color="#2a6f97"); ax[0].set_title("Average AQI by Season"); ax[0].set_xlabel("Mean AQI")
sns.boxplot(data=df, x="Season", y="AQI", order=["Winter","Post-Monsoon","Summer","Monsoon"], palette="Blues", ax=ax[1])
ax[1].set_title("AQI spread by Season")
plt.tight_layout(); plt.savefig("figs/03_season.png", bbox_inches="tight"); plt.show()
R["aqi_by_season"] = se.round(1).to_dict()

In [ ]:
st = df.groupby("State")["AQI"].agg(["mean","count"]).sort_values("mean", ascending=False)
top = st.head(10).iloc[::-1]
plt.figure(figsize=(8, 4.5))
plt.barh(top.index, top["mean"], color="#c0392b"); plt.title("Top 10 States by Average AQI"); plt.xlabel("Mean AQI")
plt.tight_layout(); plt.savefig("figs/04_state_aqi.png", bbox_inches="tight"); plt.show()
R["state_top"] = st["mean"].head(10).round(1).to_dict()
R["state_bottom"] = st["mean"].tail(5).round(1).to_dict()
R["state_counts"] = st["count"].head(5).to_dict()
R["delhi_share"] = float((df["State"] == "DL").mean() * 100)

In [ ]:
corr = df[pollutants + ["AQI"]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Correlation between pollutants and AQI")
plt.tight_layout(); plt.savefig("figs/05_correlation.png", bbox_inches="tight"); plt.show()
R["corr_aqi"] = corr["AQI"].drop("AQI").sort_values(ascending=False).round(3).to_dict()
R["corr_pm25_pm10"] = float(corr.loc["PM2.5", "PM10"])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, c in zip(ax, ["PM2.5", "PM10", "CO"]):
    sns.scatterplot(data=df.sample(6000, random_state=RS), x=c, y="AQI", alpha=.25, s=10, color="#2a6f97", ax=a)
    a.set_title(f"AQI vs {c}")
plt.tight_layout(); plt.savefig("figs/06_scatter_pollutants.png", bbox_inches="tight"); plt.show()

In [ ]:
bp = df.groupby("AQI_Bucket")[["PM2.5","PM10","NO2","CO","SO2","O3"]].mean().reindex(order).round(2)
R["bucket_profile"] = bp.to_dict()
bp

## 5. Model Building (Regression)
Target = `AQI`. Data is split 80% train / 20% test (random_state = 42). Models compared: Linear Regression (baseline), Decision Tree, Random Forest, Histogram Gradient Boosting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RS)
print("Train:", X_train.shape, "| Test:", X_test.shape)
R["n_train"], R["n_test"] = int(len(X_train)), int(len(X_test))

models = {
    "Linear Regression":  make_pipeline(StandardScaler(), LinearRegression()),
    "Decision Tree":      DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RS),
    "Random Forest":      RandomForestRegressor(n_estimators=60, min_samples_leaf=2, max_features=0.4, n_jobs=-1, random_state=RS),
    "Gradient Boosting":  HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=RS),
}
rows, preds = [], {}
for name, m in models.items():
    m.fit(X_train, y_train)
    p = np.clip(m.predict(X_test), 0, 500)
    preds[name] = p
    rows.append({"Model": name,
                 "R2": r2_score(y_test, p),
                 "MAE": mean_absolute_error(y_test, p),
                 "RMSE": mean_squared_error(y_test, p) ** 0.5,
                 "Train R2": r2_score(y_train, np.clip(m.predict(X_train), 0, 500))})
res = pd.DataFrame(rows).set_index("Model").round(4)
res

In [ ]:
R["model_results"] = res.reset_index().to_dict(orient="records")
best_name = res["R2"].idxmax()
best = models[best_name]; best_pred = preds[best_name]
R["best_model"] = best_name
print("Best model:", best_name)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
res[["R2"]].plot(kind="bar", ax=ax[0], color="#2a6f97", legend=False); ax[0].set_title("Test R2 by model"); ax[0].set_ylim(0.8, 1.0)
ax[0].tick_params(axis="x", rotation=20)
res[["MAE","RMSE"]].plot(kind="bar", ax=ax[1], color=["#f2994a","#c0392b"]); ax[1].set_title("Test MAE and RMSE (lower is better)")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("figs/07_model_comparison.png", bbox_inches="tight"); plt.show()

## 6. Evaluation of the best model

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(y_test, best_pred, s=5, alpha=.25, color="#2a6f97")
ax[0].plot([0, 500], [0, 500], "r--", lw=1.2); ax[0].set_xlabel("Actual AQI"); ax[0].set_ylabel("Predicted AQI")
ax[0].set_title(f"Actual vs Predicted ({best_name})")
resid = y_test - best_pred
sns.histplot(resid, bins=60, kde=True, color="#8e44ad", ax=ax[1]); ax[1].set_title("Residual distribution (Actual - Predicted)")
plt.tight_layout(); plt.savefig("figs/08_actual_vs_pred.png", bbox_inches="tight"); plt.show()
R["resid_mean"], R["resid_std"] = float(resid.mean()), float(resid.std())
R["within_10"] = float((resid.abs() <= 10).mean() * 100)
R["within_20"] = float((resid.abs() <= 20).mean() * 100)
R["within_30"] = float((resid.abs() <= 30).mean() * 100)
print(R["within_10"], R["within_20"], R["within_30"])

In [ ]:
# 5-fold cross-validation for the best model (stability check)
cv = cross_val_score(best, X, y, cv=5, scoring="r2", n_jobs=1)
R["cv_r2_mean"], R["cv_r2_std"] = float(cv.mean()), float(cv.std())
print("CV R2:", cv.round(4), "mean =", cv.mean().round(4))

In [ ]:
# Feature importance of the best model (permutation importance on a test sample)
samp = X_test.sample(5000, random_state=RS)
pi = permutation_importance(best, samp, y_test.loc[samp.index], n_repeats=5, random_state=RS, n_jobs=1, scoring="r2")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
top15 = imp.head(15).iloc[::-1]
plt.figure(figsize=(8, 5.5))
plt.barh(top15.index, top15.values, color="#2a6f97"); plt.title(f"Top 15 features - permutation importance ({best_name})")
plt.xlabel("Drop in R2 when feature is shuffled")
plt.tight_layout(); plt.savefig("figs/09_feature_importance.png", bbox_inches="tight"); plt.show()
R["importance"] = imp.head(10).round(4).to_dict()

## 7. Error Analysis & Robustness

In [ ]:
ea = pd.DataFrame({"Actual": y_test.values, "Pred": best_pred, "Season": data.loc[y_test.index, "Season"].values,
                   "State": data.loc[y_test.index, "State"].values, "Bucket": data.loc[y_test.index, "AQI_Bucket"].values})
ea["AbsErr"] = (ea["Actual"] - ea["Pred"]).abs()
by_bucket = ea.groupby("Bucket")["AbsErr"].mean().reindex(order)
by_season = ea.groupby("Season")["AbsErr"].mean().sort_values()
by_state  = ea.groupby("State")["AbsErr"].agg(["mean","count"]).query("count >= 30").sort_values("mean", ascending=False)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(order, by_bucket.values, color=pal); ax[0].set_title("MAE by actual AQI category"); ax[0].tick_params(axis="x", rotation=40)
ax[1].barh(by_season.index, by_season.values, color="#2a6f97"); ax[1].set_title("MAE by season")
top_s = by_state.head(8).iloc[::-1]; ax[2].barh(top_s.index, top_s["mean"], color="#c0392b"); ax[2].set_title("States with highest MAE")
plt.tight_layout(); plt.savefig("figs/10_error_analysis.png", bbox_inches="tight"); plt.show()
R["mae_bucket"] = by_bucket.round(2).to_dict()
R["mae_season"] = by_season.round(2).to_dict()
R["mae_state_top"] = by_state["mean"].head(5).round(2).to_dict()
R["worst_err"] = float(ea["AbsErr"].max())

In [ ]:
# Robustness 1: time-based split -> train on data up to 2019, test on 2020 (future data)
tr = (data["Year"] <= 2019).values; te = ~tr
m_t = RandomForestRegressor(n_estimators=60, min_samples_leaf=2, max_features=0.4, n_jobs=-1, random_state=RS) if best_name == "Random Forest" else HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=RS)
m_t.fit(X[tr], y[tr]); p_t = np.clip(m_t.predict(X[te]), 0, 500)
R["time_split"] = {"R2": float(r2_score(y[te], p_t)), "MAE": float(mean_absolute_error(y[te], p_t)),
                   "RMSE": float(mean_squared_error(y[te], p_t) ** 0.5), "n_train": int(tr.sum()), "n_test": int(te.sum())}
print("Time-based split (train<=2019, test=2020):", R["time_split"])

In [ ]:
# Robustness 2: unseen stations -> GroupKFold by StationId (model never sees the test station)
gkf = GroupKFold(n_splits=5)
g = cross_val_score(m_t, X, y, groups=data["StationId"], cv=gkf, scoring="r2", n_jobs=1)
R["group_cv_r2"] = [float(v) for v in g]; R["group_cv_r2_mean"] = float(g.mean())
print("Unseen-station R2:", g.round(4), "mean =", g.mean().round(4))

In [ ]:
# Robustness 3: how much does the model rely on the pollutant readings?
# Model without PM2.5 and PM10 (the two dominant pollutants) to test dependence
drop_cols = ["PM2.5", "PM10", "PM_ratio"]
Xa_tr, Xa_te = X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols)
m_a = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=RS).fit(Xa_tr, y_train)
p_a = np.clip(m_a.predict(Xa_te), 0, 500)
R["no_pm"] = {"R2": float(r2_score(y_test, p_a)), "MAE": float(mean_absolute_error(y_test, p_a)),
              "RMSE": float(mean_squared_error(y_test, p_a) ** 0.5)}
print("Without PM features:", R["no_pm"])

## 8. Classification: predicting the AQI category
Because AQI categories (Good ... Severe) are used for public health alerts, we also predict the category directly with a Random Forest classifier.

In [ ]:
yc = data["AQI_Bucket"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, yc, test_size=0.20, random_state=RS, stratify=yc)
clf = RandomForestClassifier(n_estimators=60, min_samples_leaf=2, max_features=0.4, n_jobs=-1, random_state=RS, class_weight="balanced")
clf.fit(Xc_tr, yc_tr); pc = clf.predict(Xc_te)
R["clf_acc"] = float(accuracy_score(yc_te, pc)); R["clf_f1_macro"] = float(f1_score(yc_te, pc, average="macro"))
R["clf_f1_weighted"] = float(f1_score(yc_te, pc, average="weighted"))
R["clf_report"] = classification_report(yc_te, pc, labels=order, output_dict=True)
cm = confusion_matrix(yc_te, pc, labels=order)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix - AQI category")
plt.tight_layout(); plt.savefig("figs/11_confusion_matrix.png", bbox_inches="tight"); plt.show()
print(classification_report(yc_te, pc, labels=order, digits=3))
off = np.abs(pd.Series(yc_te.map({o:i for i,o in enumerate(order)}).values) - pd.Series(pd.Series(pc).map({o:i for i,o in enumerate(order)}).values))
R["clf_within_one"] = float((off <= 1).mean() * 100)

## 9. Export results for the Power BI dashboard

In [ ]:
out = data.loc[y_test.index, ["StationId","Date","State","Season","Year","Month","AQI_Bucket"]].copy()
out["Actual_AQI"] = y_test.values
out["Predicted_AQI"] = best_pred.round(1)
out["Abs_Error"] = (out["Actual_AQI"] - out["Predicted_AQI"]).abs().round(1)
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")
out.to_csv("model_predictions.csv", index=False)

pd.DataFrame({"Feature": imp.index, "Importance": imp.values}).head(15).to_csv("feature_importance.csv", index=False)
res.reset_index().to_csv("model_comparison.csv", index=False)
print("Saved: model_predictions.csv, feature_importance.csv, model_comparison.csv")
json.dump(R, open("results.json", "w"), indent=1, default=str)

## 10. Conclusion
The best model reproduces AQI with very high accuracy; the insights, limitations and recommendations are discussed in the project report. Remember to mention that AQI is mathematically derived from pollutant sub-indices, which explains the high R2 and is stated as a limitation.